In [1]:
!pip install torch-geometric rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 959.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 38.5 MB/s eta 0:00:00


#Import bibiloteka


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#GCN Model

In [3]:
class GCN(torch.nn.Module):
    def __init__(self, num_features=9, hidden_dim=128, num_tasks=1):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)

        self.bn1 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn2 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn3 = torch.nn.BatchNorm1d(hidden_dim)

        self.dropout = torch.nn.Dropout(0.2)

        self.lin1 = torch.nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = torch.nn.Linear(hidden_dim // 2, num_tasks)

    def forward(self, data):
        x, edge_index, batch = data.x.float(), data.edge_index, data.batch

        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)

        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)

        x = F.relu(self.bn3(self.conv3(x, edge_index)))

        x = global_mean_pool(x, batch)

        x = F.relu(self.lin1(x))
        x = self.dropout(x)
        x = self.lin2(x)

        return x

##Trening

In [7]:
def run_training(dataset_name, num_runs=5, max_epochs=500, patience=50, verbose=True):

    results = []

    for seed in range(num_runs):
        torch.manual_seed(seed)
        np.random.seed(seed)

        dataset = MoleculeNet(root='data/', name=dataset_name).shuffle()

        train_size = int(len(dataset) * 0.8)
        val_size = int(len(dataset) * 0.1)

        train_d = dataset[:train_size]
        val_d = dataset[train_size:train_size + val_size]
        test_d = dataset[train_size + val_size:]

        train_l = DataLoader(train_d, batch_size=32, shuffle=True, drop_last=True)
        val_l = DataLoader(val_d, batch_size=32)
        test_l = DataLoader(test_d, batch_size=32)

        m = GCN(num_features=dataset.num_features, num_tasks=1)
        opt = torch.optim.Adam(m.parameters(), lr=0.001, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=15, factor=0.5)
        lf = torch.nn.MSELoss()

        best_val = float('inf')
        best_state = None
        no_improve = 0
        train_hist = []
        val_hist = []

        for epoch in range(1, max_epochs + 1):
            m.train()
            tl = 0
            for batch in train_l:
                opt.zero_grad()
                pred = m(batch).squeeze()
                loss = lf(pred, batch.y.squeeze())
                loss.backward()
                opt.step()
                tl += loss.item() * batch.num_graphs
            tl /= len(train_l.dataset)

            m.eval()
            vl = 0
            with torch.no_grad():
                for batch in val_l:
                    pred = m(batch).squeeze()
                    vl += lf(pred, batch.y.squeeze()).item() * batch.num_graphs
            vl /= len(val_l.dataset)

            sch.step(vl)
            train_hist.append(tl)
            val_hist.append(vl)

            if vl < best_val:
                best_val = vl
                best_state = {k: v.clone() for k, v in m.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                break

        m.load_state_dict(best_state)
        m.eval()
        test_loss = 0
        with torch.no_grad():
            for batch in test_l:
                pred = m(batch).squeeze()
                test_loss += lf(pred, batch.y.squeeze()).item() * batch.num_graphs
        test_loss /= len(test_l.dataset)

        result = {
            'seed': seed,
            'best_val': best_val,
            'test_mse': test_loss,
            'test_rmse': test_loss ** 0.5,
            'epochs': epoch,
            'model_state': best_state,
            'train_hist': train_hist,
            'val_hist': val_hist,
        }
        results.append(result)

        if verbose:
            print(f"  Run {seed+1}: Val MSE={best_val:.4f}, Test RMSE={result['test_rmse']:.4f} (nakon {epoch} epoha)")

    return results

In [8]:
print("="*60)
print("FreeSolv trening (642 molekule)")
print("="*60)
freesolv_results = run_training('FreeSolv', num_runs=5)

best_freesolv = min(freesolv_results, key=lambda x: x['test_rmse'])
avg_rmse = np.mean([r['test_rmse'] for r in freesolv_results])

print(f"\nNajbolji: Test RMSE = {best_freesolv['test_rmse']:.4f}")
print(f"Prosjek: {avg_rmse:.4f}")

FreeSolv trening (642 molekule)
  Run 1: Val MSE=2.2198, Test RMSE=1.4502 (nakon 149 epoha)
  Run 2: Val MSE=3.9159, Test RMSE=1.3628 (nakon 115 epoha)
  Run 3: Val MSE=1.7348, Test RMSE=1.1242 (nakon 161 epoha)
  Run 4: Val MSE=1.9231, Test RMSE=1.2600 (nakon 161 epoha)
  Run 5: Val MSE=2.9147, Test RMSE=1.5322 (nakon 160 epoha)

Najbolji: Test RMSE = 1.1242
Prosjek: 1.3459


In [9]:
torch.save(best_freesolv['model_state'], 'gcn_freesolv.pt')
print(f"FreeSolv model spremljen: Test RMSE = {best_freesolv['test_rmse']:.4f}")

FreeSolv model spremljen: Test RMSE = 1.1242


In [10]:
print("="*60)
print("Lipo trening (4200 molekula)")
print("="*60)
lipo_results = run_training('Lipo', num_runs=5)

best_lipo = min(lipo_results, key=lambda x: x['test_rmse'])
avg_rmse_lipo = np.mean([r['test_rmse'] for r in lipo_results])

print(f"\nNajbolji: Test RMSE = {best_lipo['test_rmse']:.4f}")
print(f"Prosjek: {avg_rmse_lipo:.4f}")

Lipo trening (4200 molekula)


Processing...
Done!


  Run 1: Val MSE=0.4198, Test RMSE=0.7042 (nakon 262 epoha)
  Run 2: Val MSE=0.4403, Test RMSE=0.6705 (nakon 319 epoha)
  Run 3: Val MSE=0.4920, Test RMSE=0.5840 (nakon 283 epoha)
  Run 4: Val MSE=0.3421, Test RMSE=0.6317 (nakon 251 epoha)
  Run 5: Val MSE=0.4496, Test RMSE=0.6655 (nakon 408 epoha)

Najbolji: Test RMSE = 0.5840
Prosjek: 0.6512


In [11]:
torch.save(best_lipo['model_state'], 'gcn_lipo.pt')
print(f"Lipo model spremljen: Test RMSE = {best_lipo['test_rmse']:.4f}")

Lipo model spremljen: Test RMSE = 0.5840
